# CHARACTER 3D ASSETS v1 — مجسمات الشخصيتين مع الخامات والهيكل العظمي

يشغّل هذا الدفتر بناء نموذج ثلاثي الأبعاد من كل صورة مرجعية، ثم يضيف UV/خامة وSMPL22 rig ويصدر GLB وFBX وBLEND ومعاينات وفحص جودة. يعتمد TripoSR للاستدلال المحلي، وMotius template rig محليًا. الصور لا تُرفع إلى خدمة rigging خارجية ولا تُحفظ في GitHub؛ تُمرر للبيئة عبر GitHub Actions Secrets مشفرة.

**مهم:** إعادة البناء من صورة أمامية واحدة تستنتج الجوانب والخلف، وقد تخطئ في الملابس والأطراف. نجاح الاختبارات البنيوية لا يعني اعتمادًا فنيًا نهائيًا؛ راجع المعاينات وتشوه المفاصل يدويًا.


## 1 — تحقق من GPU والمدخلات السرية
اضبط نوع المسرّع على T4 أو أعلى في GitHub Actions. لا تعرض هذه الخلية قيم الأسرار أو بيانات الصور.


In [ ]:
import os, torch
assert torch.cuda.is_available(), 'فعّل T4 GPU أو مسرّعًا أعلى من إعدادات GitHub Actions.'
assert os.environ.get('CHAR_01_IMAGE_B64'), 'السر CHAR_01_IMAGE_B64 غير موجود في إعدادات المستودع.'
assert os.environ.get('CHAR_02_IMAGE_B64'), 'السر CHAR_02_IMAGE_B64 غير موجود في إعدادات المستودع.'
assert os.environ.get('GH_TOKEN'), 'رمز GitHub لم يصل إلى بيئة Colab.'
print('GPU:', torch.cuda.get_device_name(0))
print('Both private reference inputs are present; values are intentionally hidden.')


## 2 — جلب المستودع وتشغيل المهمة
يستنسخ هذا القسم المستودع في `/content` ويشغّل المهمة القابلة لإعادة الاستخدام. يرفع السكربت حزمة ZIP وملفات GLB والمعاينات وتقارير QA إلى GitHub Release مستقل.


In [ ]:
import pathlib, subprocess, sys
repo = pathlib.Path('/content/AI-Creative-Studio')
if repo.exists():
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','--depth','1','https://github.com/h776117073-eng/AI-Creative-Studio.git',str(repo)],check=True)
subprocess.run([sys.executable,str(repo/'tasks'/'build_character_assets.py')],cwd=str(repo),check=True,env=os.environ.copy())
print('تم تنفيذ المهمة دون رمز خروج فاشل.')


## 3 — الملفات الناتجة

لكل شخصية: ملف GLB مدمج الخامة والهيكل، FBX للأنظمة التقليدية، ملف Blender قابل للتحرير، صورة معاينة، وJSON لتقرير التحقق. يوجد أيضًا `reverse_analysis_character_map.json` لربط SMPL22 بعلامات COCO-17 تقريبًا.

المخرجات في GitHub Release؛ الصور الأصلية ليست ضمن الإصدار.
